In [ ]:

# ============================================================
# PUBLICATION-LEVEL FANN EXPERIMENT
# GPU OPTIMIZED FOR GOOGLE COLAB
# ANN vs FractalNet vs FANN
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

from torch.amp import autocast, GradScaler

# ============================================================
# CHECK GPU
# ============================================================

print("====================================")

if torch.cuda.is_available():

    print("GPU ENABLED")
    print("DEVICE:", torch.cuda.get_device_name(0))

else:

    print("GPU NOT ENABLED")
    print("Go to Runtime -> Change runtime type -> GPU")

print("====================================")

# ============================================================
# DEVICE
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================
# HYPERPARAMETERS
# ============================================================

BATCH_SIZE = 128
EPOCHS = 100
LEARNING_RATE = 0.1

# ============================================================
# DATA AUGMENTATION
# ============================================================

transform_train = transforms.Compose([

    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),

    transforms.ToTensor(),

    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2023, 0.1994, 0.2010)
    )
])

transform_test = transforms.Compose([

    transforms.ToTensor(),

    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2023, 0.1994, 0.2010)
    )
])

# ============================================================
# CIFAR-10 DATASET
# ============================================================

trainset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform_train
)

testset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform_test
)

# ============================================================
# DATALOADER
# ============================================================

trainloader = torch.utils.data.DataLoader(
    trainset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

testloader = torch.utils.data.DataLoader(
    testset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# ============================================================
# ANN BASELINE
# ============================================================

class CNN_ANN(nn.Module):

    def __init__(self):

        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(3, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(256 * 8 * 8, 256),
            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(256, 10)
        )

    def forward(self, x):

        x = self.features(x)
        x = self.classifier(x)

        return x

# ============================================================
# FRACTAL BLOCK
# ============================================================

class FractalBlock(nn.Module):

    def __init__(self, in_c, out_c, depth):

        super().__init__()

        self.depth = depth

        self.base = nn.Sequential(

            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU()
        )

        if depth > 0:

            self.branch1 = FractalBlock(
                in_c,
                out_c,
                depth - 1
            )

            self.branch2 = FractalBlock(
                in_c,
                out_c,
                depth - 1
            )

        self.join = nn.Sequential(

            nn.Conv2d(out_c, out_c, 1),
            nn.BatchNorm2d(out_c),
            nn.ReLU()
        )

    def forward(self, x):

        if self.depth == 0:

            return self.base(x)

        y1 = self.branch1(x)
        y2 = self.branch2(x)

        y = (y1 + y2) / 2

        return self.join(y)

# ============================================================
# FRACTALNET
# ============================================================

class FractalNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.stem = nn.Sequential(

            nn.Conv2d(3, 64, 3, padding=1),
            nn.ReLU()
        )

        self.body = FractalBlock(
            64,
            128,
            depth=2
        )

        self.head = nn.Sequential(

            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, 10)
        )

    def forward(self, x):

        x = self.stem(x)
        x = self.body(x)
        x = self.head(x)

        return x

# ============================================================
# FANN BLOCK
# ============================================================

class FANNBlock(nn.Module):

    def __init__(self, in_c, out_c, depth):

        super().__init__()

        self.depth = depth

        self.base = nn.Sequential(

            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU()
        )

        if depth > 0:

            self.child1 = FANNBlock(
                in_c,
                out_c,
                depth - 1
            )

            self.child2 = FANNBlock(
                in_c,
                out_c,
                depth - 1
            )

        # ATTENTION JOIN OPERATOR

        self.attention = nn.Sequential(

            nn.AdaptiveAvgPool2d(1),

            nn.Conv2d(out_c, out_c, 1),

            nn.Sigmoid()
        )

        self.join = nn.Sequential(

            nn.Conv2d(out_c, out_c, 1),
            nn.BatchNorm2d(out_c),
            nn.ReLU()
        )

    def forward(self, x):

        if self.depth == 0:

            return self.base(x)

        y1 = self.child1(x)
        y2 = self.child2(x)

        a1 = self.attention(y1)
        a2 = self.attention(y2)

        y = a1 * y1 + a2 * y2

        return self.join(y)

# ============================================================
# FANN MODEL
# ============================================================

class FANN(nn.Module):

    def __init__(self):

        super().__init__()

        self.stem = nn.Sequential(

            nn.Conv2d(3, 64, 3, padding=1),
            nn.ReLU()
        )

        self.body = FANNBlock(
            64,
            128,
            depth=2
        )

        self.head = nn.Sequential(

            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, 10)
        )

    def forward(self, x):

        x = self.stem(x)
        x = self.body(x)
        x = self.head(x)

        return x

# ============================================================
# LOSS FUNCTION
# ============================================================

criterion = nn.CrossEntropyLoss()

# ============================================================
# TRAIN FUNCTION
# ============================================================

def train(model, loader, optimizer, scaler):

    model.train()

    total_loss = 0

    for x, y in loader:

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad()

        # MIXED PRECISION

        with autocast(device_type="cuda"):

            out = model(x)

            loss = criterion(out, y)

        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        total_loss += loss.item()

    return total_loss / len(loader)

# ============================================================
# EVALUATION
# ============================================================

def evaluate(model, loader):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for x, y in loader:

            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            out = model(x)

            pred = out.argmax(1)

            correct += (pred == y).sum().item()

            total += y.size(0)

    acc = 100 * correct / total
    err = 100 - acc

    return acc, err

# ============================================================
# RUN EXPERIMENT
# ============================================================

def run(model, name, epochs=EPOCHS):

    print("\n====================================")
    print("TRAINING:", name)
    print("====================================")

    model = model.to(device)

    optimizer = optim.SGD(

        model.parameters(),

        lr=LEARNING_RATE,

        momentum=0.9,

        weight_decay=5e-4
    )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=epochs
    )

    scaler = GradScaler("cuda")

    best_acc = 0

    for epoch in range(epochs):

        loss = train(
            model,
            trainloader,
            optimizer,
            scaler
        )

        acc, err = evaluate(
            model,
            testloader
        )

        scheduler.step()

        if acc > best_acc:

            best_acc = acc

            torch.save(
                model.state_dict(),
                f"{name}_BEST.pth"
            )

        print(
            f"Epoch {epoch:03d} | "
            f"Loss={loss:.4f} | "
            f"Acc={acc:.2f}% | "
            f"Err={err:.2f}%"
        )

    print(f"\nBEST ACC ({name}) = {best_acc:.2f}%")

    return best_acc

# ============================================================
# RUN ALL MODELS
# ============================================================

ann = CNN_ANN()

fra = FractalNet()

fan = FANN()

ann_result = run(ann, "ANN")

fra_result = run(fra, "FractalNet")

fan_result = run(fan, "FANN")

# ============================================================
# FINAL RESULTS TABLE
# ============================================================

print("\n====================================")
print("FINAL RESULTS")
print("====================================")

print(f"ANN        : {ann_result:.2f}%")
print(f"FractalNet : {fra_result:.2f}%")
print(f"FANN       : {fan_result:.2f}%")

print("====================================")


GPU ENABLED
DEVICE: Tesla T4


100%|██████████| 170M/170M [00:13<00:00, 13.1MB/s]



TRAINING: ANN
Epoch 000 | Loss=2.7226 | Acc=42.13% | Err=57.87%
Epoch 001 | Loss=1.6933 | Acc=45.82% | Err=54.18%
Epoch 002 | Loss=1.5859 | Acc=50.25% | Err=49.75%
Epoch 003 | Loss=1.4658 | Acc=54.02% | Err=45.98%
Epoch 004 | Loss=1.3736 | Acc=55.54% | Err=44.46%
Epoch 005 | Loss=1.2923 | Acc=56.67% | Err=43.33%
Epoch 006 | Loss=1.2313 | Acc=57.01% | Err=42.99%
Epoch 007 | Loss=1.2109 | Acc=62.88% | Err=37.12%
Epoch 008 | Loss=1.1731 | Acc=63.57% | Err=36.43%
Epoch 009 | Loss=1.1568 | Acc=60.88% | Err=39.12%
Epoch 010 | Loss=1.1355 | Acc=62.65% | Err=37.35%
Epoch 011 | Loss=1.1122 | Acc=66.15% | Err=33.85%
Epoch 012 | Loss=1.1009 | Acc=67.72% | Err=32.28%
Epoch 013 | Loss=1.0816 | Acc=68.69% | Err=31.31%
Epoch 014 | Loss=1.0733 | Acc=68.69% | Err=31.31%
Epoch 015 | Loss=1.0516 | Acc=68.67% | Err=31.33%
Epoch 016 | Loss=1.0445 | Acc=70.98% | Err=29.02%
Epoch 017 | Loss=1.0397 | Acc=67.30% | Err=32.70%
Epoch 018 | Loss=1.0400 | Acc=62.38% | Err=37.62%
Epoch 019 | Loss=1.0147 | Acc=69.33

In [ ]:
import numpy as np
import random

# ============================================================
# FIX RANDOM SEED
# ============================================================
def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# ============================================================
# MULTI-RUN EXPERIMENT
# ============================================================
runs = 5

ann_scores = []
fra_scores = []
fan_scores = []

for seed in [0,1,2,3,4]:

    print("\n==============================")
    print("RUN WITH SEED:", seed)
    print("==============================")

    set_seed(seed)

    ann = CNN_ANN()
    fra = FractalNet()
    fan = FANN()

    ann_acc = run(ann, "ANN", epochs=100)
    fra_acc = run(fra, "FractalNet", epochs=100)
    fan_acc = run(fan, "FANN", epochs=100)

    ann_scores.append(ann_acc)
    fra_scores.append(fra_acc)
    fan_scores.append(fan_acc)

# ============================================================
# FINAL STATISTICS
# ============================================================
print("\n====================================")
print("STATISTICAL RESULTS")
print("====================================")

print(
    f"ANN        : "
    f"{np.mean(ann_scores):.2f} ± {np.std(ann_scores):.2f}"
)

print(
    f"FractalNet : "
    f"{np.mean(fra_scores):.2f} ± {np.std(fra_scores):.2f}"
)

print(
    f"FANN       : "
    f"{np.mean(fan_scores):.2f} ± {np.std(fan_scores):.2f}"
)

print("====================================")

In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(count_params(ann))
print(count_params(fra))
print(count_params(fan))

4568842
639242
754826


In [ ]:

# ============================================================
# PUBLICATION-LEVEL FANN EXPERIMENT
# GPU OPTIMIZED FOR GOOGLE COLAB
# ANN vs FractalNet vs FANN
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

from torch.amp import autocast, GradScaler

# ============================================================
# CHECK GPU
# ============================================================

print("====================================")

if torch.cuda.is_available():

    print("GPU ENABLED")
    print("DEVICE:", torch.cuda.get_device_name(0))

else:

    print("GPU NOT ENABLED")
    print("Go to Runtime -> Change runtime type -> GPU")

print("====================================")

# ============================================================
# DEVICE
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================
# HYPERPARAMETERS
# ============================================================

BATCH_SIZE = 128
EPOCHS = 100
LEARNING_RATE = 0.1

# ============================================================
# DATA AUGMENTATION
# ============================================================

transform_train = transforms.Compose([

    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),

    transforms.ToTensor(),

    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2023, 0.1994, 0.2010)
    )
])

transform_test = transforms.Compose([

    transforms.ToTensor(),

    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2023, 0.1994, 0.2010)
    )
])

# ============================================================
# CIFAR-10 DATASET
# ============================================================

trainset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform_train
)

testset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform_test
)

# ============================================================
# DATALOADER
# ============================================================

trainloader = torch.utils.data.DataLoader(
    trainset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

testloader = torch.utils.data.DataLoader(
    testset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# ============================================================
# ANN BASELINE
# ============================================================

class CNN_ANN(nn.Module):

    def __init__(self):

        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(3, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(256 * 8 * 8, 256),
            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(256, 10)
        )

    def forward(self, x):

        x = self.features(x)
        x = self.classifier(x)

        return x

# ============================================================
# FRACTAL BLOCK
# ============================================================

class FractalBlock(nn.Module):

    def __init__(self, in_c, out_c, depth):

        super().__init__()

        self.depth = depth

        self.base = nn.Sequential(

            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU()
        )

        if depth > 0:

            self.branch1 = FractalBlock(
                in_c,
                out_c,
                depth - 1
            )

            self.branch2 = FractalBlock(
                in_c,
                out_c,
                depth - 1
            )

        self.join = nn.Sequential(

            nn.Conv2d(out_c, out_c, 1),
            nn.BatchNorm2d(out_c),
            nn.ReLU()
        )

    def forward(self, x):

        if self.depth == 0:

            return self.base(x)

        y1 = self.branch1(x)
        y2 = self.branch2(x)

        y = (y1 + y2) / 2

        return self.join(y)

# ============================================================
# FRACTALNET
# ============================================================

class FractalNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.stem = nn.Sequential(

            nn.Conv2d(3, 64, 3, padding=1),
            nn.ReLU()
        )

        self.body = FractalBlock(
            64,
            128,
            depth=2
        )

        self.head = nn.Sequential(

            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, 10)
        )

    def forward(self, x):

        x = self.stem(x)
        x = self.body(x)
        x = self.head(x)

        return x

# ============================================================
# FANN BLOCK
# ============================================================

class FANNBlock(nn.Module):

    def __init__(self, in_c, out_c, depth):

        super().__init__()

        self.depth = depth

        self.base = nn.Sequential(

            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU()
        )

        if depth > 0:

            self.child1 = FANNBlock(
                in_c,
                out_c,
                depth - 1
            )

            self.child2 = FANNBlock(
                in_c,
                out_c,
                depth - 1
            )

        # ATTENTION JOIN OPERATOR

        self.attention = nn.Sequential(

            nn.AdaptiveAvgPool2d(1),

            nn.Conv2d(out_c, out_c, 1),

            nn.Sigmoid()
        )

        self.join = nn.Sequential(

            nn.Conv2d(out_c, out_c, 1),
            nn.BatchNorm2d(out_c),
            nn.ReLU()
        )

    def forward(self, x):

        if self.depth == 0:

            return self.base(x)

        y1 = self.child1(x)
        y2 = self.child2(x)

        a1 = self.attention(y1)
        a2 = self.attention(y2)

        y = a1 * y1 + a2 * y2

        return self.join(y)

# ============================================================
# FANN MODEL
# ============================================================

class FANN(nn.Module):

    def __init__(self):

        super().__init__()

        self.stem = nn.Sequential(

            nn.Conv2d(3, 64, 3, padding=1),
            nn.ReLU()
        )

        self.body = FANNBlock(
            64,
            128,
            depth=2
        )

        self.head = nn.Sequential(

            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, 10)
        )

    def forward(self, x):

        x = self.stem(x)
        x = self.body(x)
        x = self.head(x)

        return x

# ============================================================
# LOSS FUNCTION
# ============================================================

criterion = nn.CrossEntropyLoss()

# ============================================================
# TRAIN FUNCTION
# ============================================================

def train(model, loader, optimizer, scaler):

    model.train()

    total_loss = 0

    for x, y in loader:

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad()

        # MIXED PRECISION

        with autocast(device_type="cuda"):

            out = model(x)

            loss = criterion(out, y)

        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        total_loss += loss.item()

    return total_loss / len(loader)

# ============================================================
# EVALUATION
# ============================================================

def evaluate(model, loader):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for x, y in loader:

            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            out = model(x)

            pred = out.argmax(1)

            correct += (pred == y).sum().item()

            total += y.size(0)

    acc = 100 * correct / total
    err = 100 - acc

    return acc, err

# ============================================================
# RUN EXPERIMENT
# ============================================================

def run(model, name, epochs=EPOCHS):

    print("\n====================================")
    print("TRAINING:", name)
    print("====================================")

    model = model.to(device)

    optimizer = optim.SGD(

        model.parameters(),

        lr=LEARNING_RATE,

        momentum=0.9,

        weight_decay=5e-4
    )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=epochs
    )

    scaler = GradScaler("cuda")

    best_acc = 0

    for epoch in range(epochs):

        loss = train(
            model,
            trainloader,
            optimizer,
            scaler
        )

        acc, err = evaluate(
            model,
            testloader
        )

        scheduler.step()

        if acc > best_acc:

            best_acc = acc

            torch.save(
                model.state_dict(),
                f"{name}_BEST.pth"
            )

        print(
            f"Epoch {epoch:03d} | "
            f"Loss={loss:.4f} | "
            f"Acc={acc:.2f}% | "
            f"Err={err:.2f}%"
        )

    print(f"\nBEST ACC ({name}) = {best_acc:.2f}%")

    return best_acc

# ============================================================
# RUN ALL MODELS
# ============================================================

ann = CNN_ANN()

fra = FractalNet()

fan = FANN()

ann_result = run(ann, "ANN")

fra_result = run(fra, "FractalNet")

fan_result = run(fan, "FANN")

# ============================================================
# FINAL RESULTS TABLE
# ============================================================

print("\n====================================")
print("FINAL RESULTS")
print("====================================")

print(f"ANN        : {ann_result:.2f}%")
print(f"FractalNet : {fra_result:.2f}%")
print(f"FANN       : {fan_result:.2f}%")

print("====================================")
import numpy as np
import random

# ============================================================
# FIX RANDOM SEED
# ============================================================
def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# ============================================================
# MULTI-RUN EXPERIMENT
# ============================================================
runs = 5

ann_scores = []
fra_scores = []
fan_scores = []

for seed in [0,1,2,3,4]:

    print("\n==============================")
    print("RUN WITH SEED:", seed)
    print("==============================")

    set_seed(seed)

    ann = CNN_ANN()
    fra = FractalNet()
    fan = FANN()

    ann_acc = run(ann, "ANN", epochs=100)
    fra_acc = run(fra, "FractalNet", epochs=100)
    fan_acc = run(fan, "FANN", epochs=100)

    ann_scores.append(ann_acc)
    fra_scores.append(fra_acc)
    fan_scores.append(fan_acc)

# ============================================================
# FINAL STATISTICS
# ============================================================
print("\n====================================")
print("STATISTICAL RESULTS")
print("====================================")

print(
    f"ANN        : "
    f"{np.mean(ann_scores):.2f} ± {np.std(ann_scores):.2f}"
)

print(
    f"FractalNet : "
    f"{np.mean(fra_scores):.2f} ± {np.std(fra_scores):.2f}"
)

print(
    f"FANN       : "
    f"{np.mean(fan_scores):.2f} ± {np.std(fan_scores):.2f}"
)

print("====================================")